In [ ]:
# Install required packages
print("📦 Installing required packages...")
!pip install -q wandb pytorch-msssim lpips rasterio scikit-learn tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import vgg19
import os
import pandas as pd
import json
import glob
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# Check GPU availability
print(f"\n{'='*70}")
print("GPU CONFIGURATION")
print(f"{'='*70}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda:0')
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    device = torch.device('cpu')
print(f"Using device: {device}")
print(f"{'='*70}\n")

# Set up output directory
OUTPUT_DIR = '/kaggle/working/RFB-ESRGAN-Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")

In [ ]:
import wandb

# Login with WandB API key
wandb.login(key="5424a3d65aac1662f5be82d4439aaac35046689e")
print("✓ WandB authentication successful!")

# Initialize WandB run
wandb.init(
    project="agricultural-sr",
    name="resume-160k-kaggle-dual-gpu",
    config={
        "platform": "kaggle",
        "gpus": torch.cuda.device_count(),
        "architecture": "RFB-ESRGAN",
        "start_iteration": 160000,
        "target_iteration": 200000,
        "num_rrdb": 12,
        "num_rrfdb": 6,
        "num_feat": 64,
        "batch_size": 8,  # Adjust based on GPU memory
        "lr": 1e-4,
        "lr_size": 32,
        "hr_size": 256
    },
    resume="allow"
)

print(f"\n✓ WandB run initialized: {wandb.run.name}")
print(f"  Project: {wandb.run.project}")
print(f"  URL: {wandb.run.url}")

In [ ]:
# Import visualization libraries
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Set up matplotlib for better visualizations
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

print("✓ Visualization libraries imported and configured!")

In [ ]:
# ========== RRDB (Residual in Residual Dense Block) ==========
class DenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, nf):
        super(RRDB, self).__init__()
        self.db1 = DenseBlock(nf)
        self.db2 = DenseBlock(nf)
        self.db3 = DenseBlock(nf)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x

# ========== RFB (Receptive Field Block) - Match checkpoint structure ==========
class RFB(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(RFB, self).__init__()
        # Split channels properly: for 64 channels -> 21, 21, 22 to sum to 64
        branch_channels = out_channels // 3
        remaining = out_channels - (branch_channels * 2)  # Third branch gets remainder
        
        # Use nn.Sequential with explicit indexing to match checkpoint structure
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, 1),  # index 0
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(branch_channels, branch_channels, 3, padding=1),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, 1),  # index 0
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(branch_channels, branch_channels, 3, padding=2, dilation=2),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, remaining, 1),  # index 0 - uses remaining channels
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(remaining, remaining, 3, padding=3, dilation=3),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        # Match checkpoint: conv_concat.0.weight
        self.conv_concat = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1)
        )

    def forward(self, x):
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = torch.cat([x1, x2, x3], dim=1)
        return self.conv_concat(x_cat) + x

class RRFDB(nn.Module):
    def __init__(self, nf):
        super(RRFDB, self).__init__()
        # Checkpoint has 5 RFB blocks per RRFDB
        self.rfb1 = RFB(nf, nf)
        self.rfb2 = RFB(nf, nf)
        self.rfb3 = RFB(nf, nf)
        self.rfb4 = RFB(nf, nf)
        self.rfb5 = RFB(nf, nf)

    def forward(self, x):
        out = self.rfb1(x)
        out = self.rfb2(out)
        out = self.rfb3(out)
        out = self.rfb4(out)
        out = self.rfb5(out)
        return out * 0.2 + x

# ========== GENERATOR - Match checkpoint structure ==========
class Generator(nn.Module):
    def __init__(self, num_rrdb=12, num_rrfdb=6, nf=64, scale=8):
        super(Generator, self).__init__()
        self.conv_first = nn.Conv2d(3, nf, 3, 1, 1)
        
        # RRDB blocks - named trunk_a in checkpoint
        self.trunk_a = nn.Sequential(*[RRDB(nf) for _ in range(num_rrdb)])
        
        # RRFDB blocks - named trunk_rfb in checkpoint
        self.trunk_rfb = nn.Sequential(*[RRFDB(nf) for _ in range(num_rrfdb)])
        
        # RFB upsampling - named rfb_up in checkpoint
        self.rfb_up = RFB(nf, nf)
        
        # Upsampling (32x32 -> 256x256 = 8x) - match checkpoint structure
        self.upsample = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 0
            nn.LeakyReLU(0.2, inplace=True),  # index 1
            nn.PixelShuffle(2),  # index 2
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 3
            nn.LeakyReLU(0.2, inplace=True),  # index 4
            nn.PixelShuffle(2),  # index 5
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 6
            nn.LeakyReLU(0.2, inplace=True),  # index 7
            nn.PixelShuffle(2)  # index 8
        )
        
        # Final convolutions - named conv_final in checkpoint
        self.conv_final = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1),  # index 0
            nn.LeakyReLU(0.2, inplace=True),  # index 1
            nn.Conv2d(nf, 3, 3, 1, 1)  # index 2
        )

    def forward(self, x):
        fea = self.conv_first(x)
        trunk_a_out = self.trunk_a(fea)
        trunk_rfb_out = self.trunk_rfb(trunk_a_out)
        rfb_up_out = self.rfb_up(trunk_rfb_out)
        fea = fea + rfb_up_out
        
        fea = self.upsample(fea)
        out = self.conv_final(fea)
        return out

# ========== DISCRIMINATOR ==========
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(512, 1024, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(1024, 1, 1)
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

print("✓ Model architectures defined")

In [ ]:
# Define dataset paths
DATASET_ROOT = '/kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2'
TRAIN_CSV = '/kaggle/input/label-indices/train.csv'
VAL_CSV = '/kaggle/input/label-indices/val.csv'

print(f"{'='*70}")
print("DATASET CONFIGURATION")
print(f"{'='*70}")
print(f"BigEarthNet Root: {DATASET_ROOT}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV: {VAL_CSV}")

# Verify paths exist
assert os.path.exists(DATASET_ROOT), f"Dataset not found: {DATASET_ROOT}"
print(f"✓ All paths verified")

# Build patch index
print(f"\n🗂️  Building patch index...")
root_contents = os.listdir(DATASET_ROOT)
all_dirs = [d for d in root_contents if os.path.isdir(os.path.join(DATASET_ROOT, d))]

patch_to_path = {}
for tile_dir in tqdm(all_dirs, desc="Indexing tiles", ncols=80):
    tile_path = os.path.join(DATASET_ROOT, tile_dir)
    try:
        for patch_name in os.listdir(tile_path):
            patch_path = os.path.join(tile_path, patch_name)
            if os.path.isdir(patch_path):
                patch_to_path[patch_name] = patch_path
    except:
        continue

print(f"   ✓ Indexed {len(patch_to_path):,} patches")

# Create train/val split from available patches
all_patch_names = list(patch_to_path.keys())
train_patches, val_patches = train_test_split(
    all_patch_names, test_size=0.2, random_state=42
)

train_df = pd.DataFrame({'patch_name': train_patches})
val_df = pd.DataFrame({'patch_name': val_patches})

print(f"\n✓ Dataset split:")
print(f"  Training: {len(train_df):,} patches")
print(f"  Validation: {len(val_df):,} patches")
print(f"{'='*70}\n")

In [ ]:
# TIF loading function
def load_rgb_from_tif(patch_path, target_size=256):
    """Load RGB bands from BigEarthNet TIF files."""
    import rasterio
    
    band_mapping = {'R': 'B04', 'G': 'B03', 'B': 'B02'}
    rgb_arrays = []
    
    for color, band_name in band_mapping.items():
        band_files = glob.glob(os.path.join(patch_path, f'*_{band_name}.tif'))
        if not band_files:
            raise FileNotFoundError(f"Band {band_name} not found in {patch_path}")
        
        with rasterio.open(band_files[0]) as src:
            band_data = src.read(1)
            band_data = np.clip(band_data / 10000.0 * 255, 0, 255).astype(np.uint8)
            rgb_arrays.append(band_data)
    
    rgb_image = np.stack(rgb_arrays, axis=-1)
    pil_image = Image.fromarray(rgb_image, mode='RGB')
    
    if pil_image.size != (target_size, target_size):
        pil_image = pil_image.resize((target_size, target_size), Image.BICUBIC)
    
    return pil_image

# Dataset class
class BigEarthNetDataset(Dataset):
    def __init__(self, dataframe, patch_index, lr_size=32, hr_size=256, transform=None):
        self.df = dataframe
        self.patch_index = patch_index
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patch_name = str(self.df.iloc[idx]['patch_name']).strip()
        patch_path = self.patch_index[patch_name]
        
        hr_img = load_rgb_from_tif(patch_path, target_size=self.hr_size)
        lr_img = hr_img.resize((self.lr_size, self.lr_size), Image.BICUBIC)
        
        if self.transform:
            lr_img = self.transform(lr_img)
            hr_img = self.transform(hr_img)
        
        return lr_img, hr_img

# Create datasets
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = BigEarthNetDataset(train_df, patch_to_path, transform=transform)
val_dataset = BigEarthNetDataset(val_df, patch_to_path, transform=transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✓ Datasets created!")
print(f"  Training batches: {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")

# Test loading
test_lr, test_hr = next(iter(train_loader))
print(f"\n🧪 Data loading test:")
print(f"  LR shape: {test_lr.shape}")
print(f"  HR shape: {test_hr.shape}")
print(f"  LR range: [{test_lr.min():.3f}, {test_lr.max():.3f}]")
print(f"  HR range: [{test_hr.min():.3f}, {test_hr.max():.3f}]")
print(f"\n✓ Data loading verified!")

In [ ]:
# Initialize models
print(f"\n{'='*70}")
print("INITIALIZING MODELS")
print(f"{'='*70}")

generator = Generator(
    num_rrdb=wandb.config.num_rrdb,
    num_rrfdb=wandb.config.num_rrfdb,
    nf=wandb.config.num_feat
)

discriminator = Discriminator()

# Load checkpoint
checkpoint_path = '/kaggle/input/generator-iter-160000-pth/pytorch/default/1/generator_iter_160000.pth'
print(f"\n📥 Loading checkpoint from iteration 160,000...")
print(f"   Path: {checkpoint_path}")

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}\n"
        "Please ensure 'generator-iter-160000-pth' dataset is added as input."
    )

checkpoint_state = torch.load(checkpoint_path, map_location='cpu')

# Debug: Print first 10 keys from each
print(f"\n🔍 Debug - Model keys (first 10):")
model_keys = list(generator.state_dict().keys())
for i, key in enumerate(model_keys[:10]):
    print(f"   {i+1}. {key}")

print(f"\n🔍 Debug - Checkpoint keys (first 10):")
checkpoint_keys = list(checkpoint_state.keys())
for i, key in enumerate(checkpoint_keys[:10]):
    print(f"   {i+1}. {key}")

# Try to load with strict=False to see what happens
missing_keys, unexpected_keys = generator.load_state_dict(checkpoint_state, strict=False)
print(f"\n⚠️  Missing keys: {len(missing_keys)}")
if len(missing_keys) > 0:
    print(f"   First missing: {missing_keys[0]}")
print(f"⚠️  Unexpected keys: {len(unexpected_keys)}")
if len(unexpected_keys) > 0:
    print(f"   First unexpected: {unexpected_keys[0]}")

if len(missing_keys) == 0 and len(unexpected_keys) == 0:
    print(f"✓ Checkpoint loaded successfully!")

# Setup for dual GPU
if torch.cuda.device_count() > 1:
    print(f"\n🚀 Using {torch.cuda.device_count()} GPUs with DataParallel")
    generator = nn.DataParallel(generator)
    discriminator = nn.DataParallel(discriminator)
    print(f"   Generator replicated across GPUs")
    print(f"   Discriminator replicated across GPUs")

generator = generator.to(device)
discriminator = discriminator.to(device)

# Print model info
gen_params = sum(p.numel() for p in generator.parameters()) / 1e6
disc_params = sum(p.numel() for p in discriminator.parameters()) / 1e6
print(f"\n📊 Model Statistics:")
print(f"   Generator parameters: {gen_params:.2f}M")
print(f"   Discriminator parameters: {disc_params:.2f}M")
print(f"   Total parameters: {gen_params + disc_params:.2f}M")
print(f"{'='*70}\n")

In [ ]:
# ========== VISUALIZATION TRACKER FOR TRAINING ==========

class TrainingVisualizer:
    """Real-time training visualization tracker"""
    def __init__(self, output_dir, val_loader, device):
        self.output_dir = output_dir
        self.val_loader = val_loader
        self.device = device
        
        # Create visualization directories
        self.loss_dir = os.path.join(output_dir, 'training_curves')
        self.samples_dir = os.path.join(output_dir, 'validation_samples')
        self.ndvi_dir = os.path.join(output_dir, 'ndvi_analysis')
        self.error_dir = os.path.join(output_dir, 'error_maps')
        self.roi_dir = os.path.join(output_dir, 'roi_analysis')
        
        os.makedirs(self.loss_dir, exist_ok=True)
        os.makedirs(self.samples_dir, exist_ok=True)
        os.makedirs(self.ndvi_dir, exist_ok=True)
        os.makedirs(self.error_dir, exist_ok=True)
        os.makedirs(self.roi_dir, exist_ok=True)
        
        # Loss history
        self.iterations = []
        self.g_losses = []
        self.d_losses = []
        self.g_pixel_losses = []
        self.g_perceptual_losses = []
        self.g_gan_losses = []
        
        # Fixed validation samples for time-lapse
        self.fixed_lr = None
        self.fixed_hr = None
        
        print("✓ Training visualizer initialized!")
    
    def set_fixed_samples(self, lr, hr):
        """Set fixed validation samples for epoch-wise comparison"""
        self.fixed_lr = lr[:4].to(self.device)  # Take first 4 samples
        self.fixed_hr = hr[:4].to(self.device)
    
    def update_losses(self, iteration, g_loss, d_loss, g_pixel, g_percept, g_gan):
        """Update loss history"""
        self.iterations.append(iteration)
        self.g_losses.append(g_loss)
        self.d_losses.append(d_loss)
        self.g_pixel_losses.append(g_pixel)
        self.g_perceptual_losses.append(g_percept)
        self.g_gan_losses.append(g_gan)
    
    def plot_loss_curves(self):
        """Generate comprehensive loss curve visualizations"""
        if len(self.iterations) < 10:
            return
        
        fig = plt.figure(figsize=(16, 10))
        gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)
        
        # 1. Main Generator vs Discriminator Loss
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(self.iterations, self.g_losses, label='Generator Loss', color='#2E86AB', linewidth=2)
        ax1.plot(self.iterations, self.d_losses, label='Discriminator Loss', color='#A23B72', linewidth=2)
        ax1.set_xlabel('Iteration', fontsize=11)
        ax1.set_ylabel('Loss', fontsize=11)
        ax1.set_title('Generator vs Discriminator Loss (GAN Training)', fontsize=13, fontweight='bold')
        ax1.legend(loc='best', fontsize=10)
        ax1.grid(alpha=0.3)
        
        # 2. Generator Loss Components
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.plot(self.iterations, self.g_pixel_losses, label='Pixel Loss (L1)', color='#F18F01', linewidth=1.5)
        ax2.plot(self.iterations, self.g_perceptual_losses, label='Perceptual Loss', color='#C73E1D', linewidth=1.5)
        ax2.plot(self.iterations, self.g_gan_losses, label='GAN Loss', color='#6A994E', linewidth=1.5)
        ax2.set_xlabel('Iteration', fontsize=10)
        ax2.set_ylabel('Loss Value', fontsize=10)
        ax2.set_title('Generator Loss Components', fontsize=11, fontweight='bold')
        ax2.legend(loc='best', fontsize=9)
        ax2.grid(alpha=0.3)
        
        # 3. Smoothed Generator Loss
        ax3 = fig.add_subplot(gs[1, 1])
        window = min(50, len(self.g_losses) // 10)
        if window > 1:
            smoothed = np.convolve(self.g_losses, np.ones(window)/window, mode='valid')
            smoothed_iters = self.iterations[window-1:]
            ax3.plot(smoothed_iters, smoothed, color='#2E86AB', linewidth=2)
        ax3.set_xlabel('Iteration', fontsize=10)
        ax3.set_ylabel('Smoothed G Loss', fontsize=10)
        ax3.set_title(f'Generator Loss (Smoothed, window={window})', fontsize=11, fontweight='bold')
        ax3.grid(alpha=0.3)
        
        # 4. Loss Ratio (Stability Indicator)
        ax4 = fig.add_subplot(gs[2, 0])
        ratio = np.array(self.g_losses) / (np.array(self.d_losses) + 1e-8)
        ax4.plot(self.iterations, ratio, color='#9B59B6', linewidth=1.5)
        ax4.axhline(y=1.0, color='red', linestyle='--', linewidth=1, label='Balance (G/D=1)')
        ax4.set_xlabel('Iteration', fontsize=10)
        ax4.set_ylabel('G Loss / D Loss', fontsize=10)
        ax4.set_title('GAN Balance Ratio (Stability Indicator)', fontsize=11, fontweight='bold')
        ax4.legend(loc='best', fontsize=9)
        ax4.grid(alpha=0.3)
        
        # 5. Recent Loss Trend (Last 500 iterations)
        ax5 = fig.add_subplot(gs[2, 1])
        recent_n = min(500, len(self.iterations))
        if recent_n > 10:
            ax5.plot(self.iterations[-recent_n:], self.g_losses[-recent_n:], 
                    label='Generator', color='#2E86AB', linewidth=2)
            ax5.plot(self.iterations[-recent_n:], self.d_losses[-recent_n:], 
                    label='Discriminator', color='#A23B72', linewidth=2)
            ax5.set_xlabel('Iteration', fontsize=10)
            ax5.set_ylabel('Loss', fontsize=10)
            ax5.set_title(f'Recent Training Progress (Last {recent_n} iters)', fontsize=11, fontweight='bold')
            ax5.legend(loc='best', fontsize=9)
            ax5.grid(alpha=0.3)
        
        # Save figure
        save_path = os.path.join(self.loss_dir, f'loss_curves_iter_{self.iterations[-1]}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # Also upload to wandb
        wandb.log({'training/loss_curves': wandb.Image(save_path)})
    
    def generate_validation_samples(self, generator, iteration, epoch=None):
        """Generate epoch-wise validation samples showing model evolution"""
        if self.fixed_lr is None:
            return
        
        generator.eval()
        with torch.no_grad():
            sr_output = generator(self.fixed_lr)
        
        # Create 4-panel comparison for each sample
        fig, axes = plt.subplots(4, 4, figsize=(18, 18))
        
        for i in range(4):
            # Low-res input (upsampled for visualization)
            lr_up = F.interpolate(self.fixed_lr[i:i+1], scale_factor=8, mode='nearest')
            lr_np = lr_up[0].cpu().permute(1, 2, 0).numpy()
            lr_np = (lr_np + 1) / 2  # Denormalize from [-1,1] to [0,1]
            lr_np = np.clip(lr_np, 0, 1)
            axes[i, 0].imshow(lr_np)
            axes[i, 0].set_title(f'Sample {i+1}: LR Input (10m)', fontsize=10)
            axes[i, 0].axis('off')
            
            # Bicubic interpolation
            bicubic = F.interpolate(self.fixed_lr[i:i+1], scale_factor=8, mode='bicubic')
            bicubic_np = bicubic[0].cpu().permute(1, 2, 0).numpy()
            bicubic_np = (bicubic_np + 1) / 2
            bicubic_np = np.clip(bicubic_np, 0, 1)
            axes[i, 1].imshow(bicubic_np)
            axes[i, 1].set_title('Bicubic Upsampling', fontsize=10)
            axes[i, 1].axis('off')
            
            # RFB-ESRGAN output
            sr_np = sr_output[i].cpu().permute(1, 2, 0).numpy()
            sr_np = (sr_np + 1) / 2
            sr_np = np.clip(sr_np, 0, 1)
            axes[i, 2].imshow(sr_np)
            title_text = f'RFB-ESRGAN (2.5m)\nIter: {iteration:,}'
            if epoch is not None:
                title_text = f'RFB-ESRGAN (2.5m)\nEpoch {epoch}, Iter: {iteration:,}'
            axes[i, 2].set_title(title_text, fontsize=10, color='green', fontweight='bold')
            axes[i, 2].axis('off')
            
            # Ground truth
            hr_np = self.fixed_hr[i].cpu().permute(1, 2, 0).numpy()
            hr_np = (hr_np + 1) / 2
            hr_np = np.clip(hr_np, 0, 1)
            axes[i, 3].imshow(hr_np)
            axes[i, 3].set_title('Ground Truth (2.5m)', fontsize=10)
            axes[i, 3].axis('off')
        
        epoch_str = f'_epoch_{epoch}' if epoch is not None else ''
        plt.suptitle(f'Validation Samples - Iteration {iteration:,}{epoch_str}', 
                    fontsize=14, fontweight='bold', y=0.995)
        plt.tight_layout()
        
        save_path = os.path.join(self.samples_dir, f'val_samples_iter_{iteration}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # Upload to wandb
        wandb.log({
            'training/validation_samples': wandb.Image(save_path),
            'iteration': iteration
        })
        
        generator.train()
    
    def generate_ndvi_comparison(self, generator, iteration):
        """Generate NDVI spectral consistency analysis"""
        if self.fixed_lr is None:
            return
        
        generator.eval()
        with torch.no_grad():
            sr_output = generator(self.fixed_lr[:2])  # First 2 samples
        
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        
        for i in range(2):
            # Extract red and NIR approximations
            # Assuming RGB channels, we'll use R and G as proxies
            sr_np = sr_output[i].cpu().numpy()
            hr_np = self.fixed_hr[i].cpu().numpy()
            
            # Simple NDVI calculation (using R and G channels as proxy)
            sr_red = sr_np[0]
            sr_nir = sr_np[1]  # Using green as NIR proxy
            sr_ndvi = (sr_nir - sr_red) / (sr_nir + sr_red + 1e-8)
            
            hr_red = hr_np[0]
            hr_nir = hr_np[1]
            hr_ndvi = (hr_nir - hr_red) / (hr_nir + hr_red + 1e-8)
            
            # Normalize for display
            sr_np_vis = (sr_np.transpose(1, 2, 0) + 1) / 2
            hr_np_vis = (hr_np.transpose(1, 2, 0) + 1) / 2
            
            # Plot RGB images
            axes[i, 0].imshow(np.clip(sr_np_vis, 0, 1))
            axes[i, 0].set_title(f'Sample {i+1}: SR Output', fontsize=10)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(np.clip(hr_np_vis, 0, 1))
            axes[i, 1].set_title('Ground Truth', fontsize=10)
            axes[i, 1].axis('off')
            
            # Plot NDVI maps
            im1 = axes[i, 2].imshow(sr_ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
            axes[i, 2].set_title('SR NDVI', fontsize=10)
            axes[i, 2].axis('off')
            plt.colorbar(im1, ax=axes[i, 2], fraction=0.046)
            
            im2 = axes[i, 3].imshow(hr_ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
            axes[i, 3].set_title('GT NDVI', fontsize=10)
            axes[i, 3].axis('off')
            plt.colorbar(im2, ax=axes[i, 3], fraction=0.046)
        
        plt.suptitle(f'NDVI Spectral Consistency - Iteration {iteration:,}', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_path = os.path.join(self.ndvi_dir, f'ndvi_iter_{iteration}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        wandb.log({
            'training/ndvi_analysis': wandb.Image(save_path),
            'iteration': iteration
        })
        
        generator.train()
    
    def generate_error_maps(self, generator, iteration):
        """Generate pixel difference heatmaps"""
        if self.fixed_lr is None:
            return
        
        generator.eval()
        with torch.no_grad():
            sr_output = generator(self.fixed_lr[:2])
        
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        
        for i in range(2):
            sr_np = sr_output[i].cpu().numpy()
            hr_np = self.fixed_hr[i].cpu().numpy()
            
            # Calculate absolute difference
            diff = np.abs(sr_np - hr_np)
            diff_magnitude = np.mean(diff, axis=0)  # Average across channels
            
            # Visualizations
            sr_vis = (sr_np.transpose(1, 2, 0) + 1) / 2
            hr_vis = (hr_np.transpose(1, 2, 0) + 1) / 2
            
            axes[i, 0].imshow(np.clip(sr_vis, 0, 1))
            axes[i, 0].set_title(f'Sample {i+1}: SR Output', fontsize=10)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(np.clip(hr_vis, 0, 1))
            axes[i, 1].set_title('Ground Truth', fontsize=10)
            axes[i, 1].axis('off')
            
            im = axes[i, 2].imshow(diff_magnitude, cmap='hot', vmin=0, vmax=0.5)
            axes[i, 2].set_title('Error Heatmap (bright = high error)', fontsize=10, color='red')
            axes[i, 2].axis('off')
            plt.colorbar(im, ax=axes[i, 2], fraction=0.046)
        
        plt.suptitle(f'Pixel Difference Error Maps - Iteration {iteration:,}', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_path = os.path.join(self.error_dir, f'error_map_iter_{iteration}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        wandb.log({
            'training/error_maps': wandb.Image(save_path),
            'iteration': iteration
        })
        
        generator.train()
    
    def generate_roi_analysis(self, generator, iteration):
        """Generate ROI zoom-in analysis for agricultural features"""
        if self.fixed_lr is None:
            return
        
        generator.eval()
        with torch.no_grad():
            sr_output = generator(self.fixed_lr[:1])  # Just first sample
        
        sr_np = sr_output[0].cpu().numpy()
        hr_np = self.fixed_hr[0].cpu().numpy()
        
        # Denormalize
        sr_vis = (sr_np.transpose(1, 2, 0) + 1) / 2
        hr_vis = (hr_np.transpose(1, 2, 0) + 1) / 2
        sr_vis = np.clip(sr_vis, 0, 1)
        hr_vis = np.clip(hr_vis, 0, 1)
        
        # Define ROI regions (crops from different areas)
        h, w = sr_vis.shape[:2]
        roi_size = 64
        
        # Define 3 ROIs: top-left, center, bottom-right
        rois = [
            (h//4, w//4),           # Top-left region
            (h//2, w//2),           # Center
            (3*h//4, 3*w//4)        # Bottom-right
        ]
        
        fig, axes = plt.subplots(3, 3, figsize=(14, 14))
        
        for i, (cy, cx) in enumerate(rois):
            # Calculate crop bounds
            y1 = max(0, cy - roi_size//2)
            y2 = min(h, cy + roi_size//2)
            x1 = max(0, cx - roi_size//2)
            x2 = min(w, cx + roi_size//2)
            
            # Extract ROIs
            sr_roi = sr_vis[y1:y2, x1:x2]
            hr_roi = hr_vis[y1:y2, x1:x2]
            
            # Calculate sharpness (edge strength)
            sr_gray = np.mean(sr_roi, axis=2)
            hr_gray = np.mean(hr_roi, axis=2)
            
            from scipy.ndimage import laplace
            sr_edges = np.abs(laplace(sr_gray))
            hr_edges = np.abs(laplace(hr_gray))
            
            # Plot SR ROI
            axes[i, 0].imshow(sr_roi)
            axes[i, 0].set_title(f'ROI {i+1}: SR Output', fontsize=10)
            axes[i, 0].axis('off')
            
            # Plot GT ROI
            axes[i, 1].imshow(hr_roi)
            axes[i, 1].set_title(f'Ground Truth', fontsize=10)
            axes[i, 1].axis('off')
            
            # Plot edge comparison
            axes[i, 2].imshow(sr_edges, cmap='gray')
            axes[i, 2].set_title(f'Edge Strength (SR)', fontsize=10)
            axes[i, 2].axis('off')
        
        plt.suptitle(f'ROI Feature Analysis - Iteration {iteration:,}\\n' +
                    'Examining crop rows, field boundaries, and texture fidelity',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_path = os.path.join(self.roi_dir, f'roi_analysis_iter_{iteration}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        wandb.log({
            'training/roi_analysis': wandb.Image(save_path),
            'iteration': iteration
        })
        
        generator.train()

print("✓ Training visualizer class defined!")

In [ ]:
# ========== LOSS FUNCTIONS AND TRAINING UTILITIES ==========

print(f"\n{'='*70}")
print("INITIALIZING LOSS FUNCTIONS")
print(f"{'='*70}")

# ========== Perceptual Loss (VGG19) ==========
class PerceptualLoss(nn.Module):
    """VGG19-based perceptual loss"""
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True).features[:35].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.criterion = nn.L1Loss()

    def forward(self, sr, hr):
        sr_features = self.vgg(sr)
        hr_features = self.vgg(hr)
        return self.criterion(sr_features, hr_features)


# ========== Initialize Loss Functions ==========
criterion_pixel = nn.L1Loss().to(device)
criterion_gan = nn.BCEWithLogitsLoss().to(device)
criterion_perceptual = PerceptualLoss().to(device)

# LPIPS loss
import lpips
lpips_loss_fn = lpips.LPIPS(net='alex').to(device)

# MS-SSIM (already imported in earlier cells)
from pytorch_msssim import ms_ssim

print(f"✓ Pixel Loss (L1): Initialized")
print(f"✓ GAN Loss (BCE with Logits): Initialized")
print(f"✓ Perceptual Loss (VGG19): Initialized")
print(f"✓ LPIPS Loss (AlexNet): Initialized")
print(f"✓ MS-SSIM Loss: Available")
print(f"{'='*70}\n")

print("✓ All loss functions ready for training!")

In [ ]:
def train_stage2_resume(
    generator,
    discriminator,
    train_loader,
    val_loader,
    total_iterations=200000,
    start_iter=160000,
    lr=1e-4
):
    """
    Stage 2 GAN training with dual GPU support and comprehensive visualization.
    """
    print(f"\n{'='*70}")
    print("STAGE 2: RESUMING GAN TRAINING WITH VISUALIZATION")
    print(f"{'='*70}")
    print(f"Starting from: {start_iter:,}")
    print(f"Target: {total_iterations:,}")
    print(f"Remaining: {total_iterations - start_iter:,} iterations")
    print(f"{'='*70}\n")
    
    # Initialize visualizer
    visualizer = TrainingVisualizer(OUTPUT_DIR, val_loader, device)
    
    # Set fixed validation samples for time-lapse visualization
    val_lr, val_hr = next(iter(val_loader))
    visualizer.set_fixed_samples(val_lr, val_hr)
    print("✓ Fixed validation samples set for epoch-wise tracking")
    
    # Optimizers
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.9, 0.99))
    
    for param_group in optimizer_g.param_groups:
        param_group['initial_lr'] = lr
    for param_group in optimizer_d.param_groups:
        param_group['initial_lr'] = lr
    
    # LR schedulers
    milestones = [50000, 100000, 150000, 180000]
    scheduler_g = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_g, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    scheduler_d = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_d, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    
    # Training loop
    iteration = start_iter
    epoch = 0
    
    print(f"Starting training loop with real-time visualizations...\n")
    
    while iteration < total_iterations:
        epoch += 1
        generator.train()
        discriminator.train()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}", ncols=100)
        
        for lr_imgs, hr_imgs in pbar:
            if iteration >= total_iterations:
                break
            
            lr_imgs = lr_imgs.to(device)
            hr_imgs = hr_imgs.to(device)
            batch_size = lr_imgs.size(0)
            
            # Train Discriminator
            optimizer_d.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            real_validity = discriminator(hr_imgs)
            fake_validity = discriminator(sr_imgs.detach())
            
            real_labels = torch.ones_like(real_validity)
            fake_labels = torch.zeros_like(fake_validity)
            
            d_loss_real = criterion_gan(real_validity, real_labels)
            d_loss_fake = criterion_gan(fake_validity, fake_labels)
            d_loss = (d_loss_real + d_loss_fake) / 2
            
            d_loss.backward()
            optimizer_d.step()
            
            # Train Generator
            optimizer_g.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            fake_validity = discriminator(sr_imgs)
            
            # Generator losses
            g_loss_pixel = criterion_pixel(sr_imgs, hr_imgs)
            g_loss_perceptual = criterion_perceptual(sr_imgs, hr_imgs)
            g_loss_gan = criterion_gan(fake_validity, torch.ones_like(fake_validity))
            
            lpips_val = lpips_loss_fn(sr_imgs, hr_imgs).mean()
            ms_ssim_val = 1 - ms_ssim(sr_imgs, hr_imgs, data_range=2.0, size_average=True)
            
            g_loss = (
                g_loss_pixel * 1.0 +
                g_loss_perceptual * 0.1 +
                g_loss_gan * 0.005 +
                lpips_val * 0.1 +
                ms_ssim_val * 0.1
            )
            
            g_loss.backward()
            optimizer_g.step()
            
            iteration += 1
            
            # Update visualizer loss history
            visualizer.update_losses(
                iteration, 
                g_loss.item(), 
                d_loss.item(),
                g_loss_pixel.item(),
                g_loss_perceptual.item(),
                g_loss_gan.item()
            )
            
            # Update progress bar
            pbar.set_postfix({
                'iter': f'{iteration}',
                'g_loss': f'{g_loss.item():.4f}',
                'd_loss': f'{d_loss.item():.4f}'
            })
            
            # Log to WandB
            if iteration % 100 == 0:
                wandb.log({
                    'iteration': iteration,
                    'g_loss': g_loss.item(),
                    'd_loss': d_loss.item(),
                    'g_loss_pixel': g_loss_pixel.item(),
                    'g_loss_perceptual': g_loss_perceptual.item(),
                    'g_loss_gan': g_loss_gan.item(),
                    'lpips': lpips_val.item(),
                    'ms_ssim': ms_ssim_val.item(),
                    'lr_g': optimizer_g.param_groups[0]['lr'],
                    'lr_d': optimizer_d.param_groups[0]['lr']
                })
            
            # ========== VISUALIZATION OUTPUTS ==========
            
            # Generate loss curves every 500 iterations
            if iteration % 500 == 0:
                print(f"\n📊 Generating loss curves...")
                visualizer.plot_loss_curves()
            
            # Generate validation samples every 1000 iterations
            if iteration % 1000 == 0:
                print(f"\n🖼️  Generating validation samples...")
                visualizer.generate_validation_samples(generator, iteration, epoch)
            
            # Generate error maps every 2000 iterations
            if iteration % 2000 == 0:
                print(f"\n🔥 Generating error heatmaps...")
                visualizer.generate_error_maps(generator, iteration)
            
            # Generate NDVI analysis every 2000 iterations
            if iteration % 2000 == 0:
                print(f"\n🌿 Generating NDVI spectral consistency analysis...")
                visualizer.generate_ndvi_comparison(generator, iteration)
            
            # Generate ROI analysis every 3000 iterations
            if iteration % 3000 == 0:
                print(f"\n🔍 Generating ROI feature analysis...")
                visualizer.generate_roi_analysis(generator, iteration)
            
            # Save checkpoint
            if iteration % 5000 == 0:
                save_path = os.path.join(OUTPUT_DIR, f'generator_iter_{iteration}.pth')
                if isinstance(generator, nn.DataParallel):
                    torch.save(generator.module.state_dict(), save_path)
                else:
                    torch.save(generator.state_dict(), save_path)
                print(f"\n💾 Checkpoint saved: {save_path}")
            
            # Validation
            if iteration % 1000 == 0:
                generator.eval()
                val_losses = []
                
                with torch.no_grad():
                    for val_lr, val_hr in val_loader:
                        val_lr = val_lr.to(device)
                        val_hr = val_hr.to(device)
                        val_sr = generator(val_lr)
                        val_loss = criterion_pixel(val_sr, val_hr)
                        val_losses.append(val_loss.item())
                
                avg_val_loss = np.mean(val_losses)
                wandb.log({'val_loss': avg_val_loss, 'iteration': iteration})
                print(f"\n📊 Validation loss: {avg_val_loss:.4f}")
                
                generator.train()
        
        scheduler_g.step()
        scheduler_d.step()
        
        print(f"\n✅ Epoch {epoch} completed!")
    
    print(f"\n{'='*70}")
    print("✅ TRAINING COMPLETED WITH COMPREHENSIVE VISUALIZATIONS!")
    print(f"{'='*70}")
    print(f"Final iteration: {iteration:,}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"  • Loss curves: {visualizer.loss_dir}")
    print(f"  • Validation samples: {visualizer.samples_dir}")
    print(f"  • NDVI analysis: {visualizer.ndvi_dir}")
    print(f"  • Error maps: {visualizer.error_dir}")
    print(f"  • ROI analysis: {visualizer.roi_dir}")
    print(f"{'='*70}\n")

print("✓ Enhanced training function with visualization defined")

In [ ]:
def train_stage2_resume(
    generator,
    discriminator,
    train_loader,
    val_loader,
    total_iterations=200000,
    start_iter=160000,
    lr=1e-4
):
    """
    Stage 2 GAN training with dual GPU support.
    """
    print(f"\n{'='*70}")
    print("STAGE 2: RESUMING GAN TRAINING")
    print(f"{'='*70}")
    print(f"Starting from: {start_iter:,}")
    print(f"Target: {total_iterations:,}")
    print(f"Remaining: {total_iterations - start_iter:,} iterations")
    print(f"{'='*70}\n")
    
    # Optimizers
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.9, 0.99))
    
    for param_group in optimizer_g.param_groups:
        param_group['initial_lr'] = lr
    for param_group in optimizer_d.param_groups:
        param_group['initial_lr'] = lr
    
    # LR schedulers
    milestones = [50000, 100000, 150000, 180000]
    scheduler_g = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_g, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    scheduler_d = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_d, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    
    # Training loop
    iteration = start_iter
    epoch = 0
    
    print(f"Starting training loop...\n")
    
    while iteration < total_iterations:
        epoch += 1
        generator.train()
        discriminator.train()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}", ncols=100)
        
        for lr_imgs, hr_imgs in pbar:
            if iteration >= total_iterations:
                break
            
            lr_imgs = lr_imgs.to(device)
            hr_imgs = hr_imgs.to(device)
            batch_size = lr_imgs.size(0)
            
            # Train Discriminator
            optimizer_d.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            real_validity = discriminator(hr_imgs)
            fake_validity = discriminator(sr_imgs.detach())
            
            real_labels = torch.ones_like(real_validity)
            fake_labels = torch.zeros_like(fake_validity)
            
            d_loss_real = criterion_gan(real_validity, real_labels)
            d_loss_fake = criterion_gan(fake_validity, fake_labels)
            d_loss = (d_loss_real + d_loss_fake) / 2
            
            d_loss.backward()
            optimizer_d.step()
            
            # Train Generator
            optimizer_g.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            fake_validity = discriminator(sr_imgs)
            
            # Generator losses
            g_loss_pixel = criterion_pixel(sr_imgs, hr_imgs)
            g_loss_perceptual = criterion_perceptual(sr_imgs, hr_imgs)
            g_loss_gan = criterion_gan(fake_validity, torch.ones_like(fake_validity))
            
            lpips_val = lpips_loss_fn(sr_imgs, hr_imgs).mean()
            ms_ssim_val = 1 - ms_ssim(sr_imgs, hr_imgs, data_range=2.0, size_average=True)
            
            g_loss = (
                g_loss_pixel * 1.0 +
                g_loss_perceptual * 0.1 +
                g_loss_gan * 0.005 +
                lpips_val * 0.1 +
                ms_ssim_val * 0.1
            )
            
            g_loss.backward()
            optimizer_g.step()
            
            iteration += 1
            
            # Update progress bar
            pbar.set_postfix({
                'iter': f'{iteration}',
                'g_loss': f'{g_loss.item():.4f}',
                'd_loss': f'{d_loss.item():.4f}'
            })
            
            # Log to WandB
            if iteration % 100 == 0:
                wandb.log({
                    'iteration': iteration,
                    'g_loss': g_loss.item(),
                    'd_loss': d_loss.item(),
                    'g_loss_pixel': g_loss_pixel.item(),
                    'g_loss_perceptual': g_loss_perceptual.item(),
                    'g_loss_gan': g_loss_gan.item(),
                    'lpips': lpips_val.item(),
                    'ms_ssim': ms_ssim_val.item(),
                    'lr_g': optimizer_g.param_groups[0]['lr'],
                    'lr_d': optimizer_d.param_groups[0]['lr']
                })
            
            # Save checkpoint
            if iteration % 5000 == 0:
                save_path = os.path.join(OUTPUT_DIR, f'generator_iter_{iteration}.pth')
                if isinstance(generator, nn.DataParallel):
                    torch.save(generator.module.state_dict(), save_path)
                else:
                    torch.save(generator.state_dict(), save_path)
                print(f"\n💾 Checkpoint saved: {save_path}")
            
            # Validation
            if iteration % 1000 == 0:
                generator.eval()
                val_losses = []
                
                with torch.no_grad():
                    for val_lr, val_hr in val_loader:
                        val_lr = val_lr.to(device)
                        val_hr = val_hr.to(device)
                        val_sr = generator(val_lr)
                        val_loss = criterion_pixel(val_sr, val_hr)
                        val_losses.append(val_loss.item())
                
                avg_val_loss = np.mean(val_losses)
                wandb.log({'val_loss': avg_val_loss, 'iteration': iteration})
                print(f"\n📊 Validation loss: {avg_val_loss:.4f}")
                
                generator.train()
        
        scheduler_g.step()
        scheduler_d.step()
    
    print(f"\n{'='*70}")
    print("✅ TRAINING COMPLETED!")
    print(f"{'='*70}")
    print(f"Final iteration: {iteration:,}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

print("✓ Training function defined")

In [ ]:
# Start training
train_stage2_resume(
    generator=generator,
    discriminator=discriminator,
    train_loader=train_loader,
    val_loader=val_loader,
    total_iterations=200000,
    start_iter=160000,
    lr=1e-4
)

In [ ]:
# Save final model
final_path = os.path.join(OUTPUT_DIR, 'generator_final_200000.pth')

if isinstance(generator, nn.DataParallel):
    torch.save(generator.module.state_dict(), final_path)
else:
    torch.save(generator.state_dict(), final_path)

print(f"\n✅ Final model saved: {final_path}")
print(f"\n📥 Download from: /kaggle/working/RFB-ESRGAN-Output/")

# Close WandB
wandb.finish()
print("\n✓ WandB run completed")

In [ ]:
# ========== COMPREHENSIVE EVALUATION METRICS ==========

# Install additional packages if needed
!pip install -q lpips pytorch-msssim scikit-learn seaborn scipy

import time
from collections import defaultdict
import seaborn as sns

print("✓ Evaluation packages installed!")

# ========== 1. SUPER-RESOLUTION METRICS ==========

class SuperResolutionMetrics:
    """Comprehensive SR evaluation metrics"""
    def __init__(self, device):
        self.device = device
        # LPIPS loss network (Alex)
        import lpips
        self.lpips_fn = lpips.LPIPS(net='alex').to(device)

    def calculate_psnr(self, sr, hr):
        """Peak Signal-to-Noise Ratio"""
        mse = F.mse_loss(sr, hr)
        psnr = 10 * torch.log10(4 / mse)  # Range [-1,1] → max=2, so 4
        return psnr.item()

    def calculate_ssim(self, sr, hr):
        """Structural Similarity Index"""
        from pytorch_msssim import ssim
        # Normalize from [-1,1] to [0,1]
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ssim_val = ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ssim_val.item()

    def calculate_ms_ssim(self, sr, hr):
        """Multi-Scale Structural Similarity Index"""
        from pytorch_msssim import ms_ssim
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ms_ssim_val = ms_ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ms_ssim_val.item()

    def calculate_lpips(self, sr, hr):
        """Learned Perceptual Image Patch Similarity"""
        lpips_val = self.lpips_fn(sr, hr)
        return lpips_val.mean().item()

    def calculate_mae(self, sr, hr):
        """Mean Absolute Error"""
        mae = F.l1_loss(sr, hr)
        return mae.item()

    def calculate_rmse(self, sr, hr):
        """Root Mean Square Error"""
        mse = F.mse_loss(sr, hr)
        rmse = torch.sqrt(mse)
        return rmse.item()

    def calculate_ndvi_error(self, sr, hr):
        """Spectral Consistency - NDVI Error for vegetation index accuracy"""
        # Extract red channel (assuming channel 0 is red after normalization)
        sr_red = sr[:, 0:1, :, :]  # Red channel
        hr_red = hr[:, 0:1, :, :]
        # Simplified NDVI approximation
        ndvi_error = F.l1_loss(sr_red, hr_red)
        return ndvi_error.item()

    def calculate_edge_preservation(self, sr, hr):
        """Edge Preservation using Sobel filters"""
        # Simple edge detection using convolution
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)

        # Average across channels
        sr_gray = sr.mean(dim=1, keepdim=True)
        hr_gray = hr.mean(dim=1, keepdim=True)

        # Apply Sobel filters
        sr_edge_x = F.conv2d(sr_gray, sobel_x, padding=1)
        sr_edge_y = F.conv2d(sr_gray, sobel_y, padding=1)
        hr_edge_x = F.conv2d(hr_gray, sobel_x, padding=1)
        hr_edge_y = F.conv2d(hr_gray, sobel_y, padding=1)

        sr_edge = torch.sqrt(sr_edge_x**2 + sr_edge_y**2)
        hr_edge = torch.sqrt(hr_edge_x**2 + hr_edge_y**2)

        edge_error = F.l1_loss(sr_edge, hr_edge)
        return edge_error.item()

    def evaluate_batch(self, sr, hr):
        """Evaluate all SR metrics on a batch"""
        metrics = {
            'psnr': self.calculate_psnr(sr, hr),
            'ssim': self.calculate_ssim(sr, hr),
            'ms_ssim': self.calculate_ms_ssim(sr, hr),
            'lpips': self.calculate_lpips(sr, hr),
            'mae': self.calculate_mae(sr, hr),
            'rmse': self.calculate_rmse(sr, hr),
            'ndvi_error': self.calculate_ndvi_error(sr, hr),
            'edge_preservation': self.calculate_edge_preservation(sr, hr)
        }
        return metrics


# ========== 2. BASELINE COMPARISON MODELS ==========

class BicubicUpsampler:
    """Baseline bicubic interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)


class BilinearUpsampler:
    """Baseline bilinear interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bilinear', align_corners=False)


class NearestUpsampler:
    """Baseline nearest neighbor interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='nearest')


class SimpleSRCNN(nn.Module):
    """Lightweight SRCNN baseline for comparison"""
    def __init__(self, scale_factor=8):
        super(SimpleSRCNN, self).__init__()
        self.scale_factor = scale_factor
        # SRCNN: 3 conv layers
        self.conv1 = nn.Conv2d(3, 64, 9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, 1, padding=0)
        self.conv3 = nn.Conv2d(32, 3, 5, padding=2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Bicubic upsampling first
        x = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return torch.tanh(x)


print("✓ Evaluation metrics and baseline models defined!")

In [ ]:
# ========== 3. COMPREHENSIVE COMPARATIVE EVALUATION ==========

def comparative_evaluation(generator, val_loader, device, num_samples=100):
    """Compare RFB-ESRGAN against multiple baselines with comprehensive metrics"""
    print("\n" + "="*70)
    print("COMPARATIVE EVALUATION: RFB-ESRGAN vs. Baselines")
    print("="*70)

    # Initialize models and metrics
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    bilinear = BilinearUpsampler(scale_factor=8)
    nearest = NearestUpsampler(scale_factor=8)
    srcnn = SimpleSRCNN(scale_factor=8).to(device)
    srcnn.eval()

    # Results storage
    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    results = {name: defaultdict(list) for name in model_names}
    inference_times = {name: [] for name in model_names}

    generator.eval()
    sample_count = 0

    print(f"\n📊 Evaluating on {num_samples} samples...")
    print(f"Models: Nearest, Bilinear, Bicubic, SRCNN, RFB-ESRGAN (Ours)")

    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Evaluating", ncols=80):
            if sample_count >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            # ========== Nearest Neighbor ==========
            start_time = time.time()
            sr_nearest = nearest(lr_img)
            inference_times['nearest'].append(time.time() - start_time)
            metrics_nearest = sr_metrics.evaluate_batch(sr_nearest, hr_img)
            for k, v in metrics_nearest.items():
                results['nearest'][k].append(v)

            # ========== Bilinear ==========
            start_time = time.time()
            sr_bilinear = bilinear(lr_img)
            inference_times['bilinear'].append(time.time() - start_time)
            metrics_bilinear = sr_metrics.evaluate_batch(sr_bilinear, hr_img)
            for k, v in metrics_bilinear.items():
                results['bilinear'][k].append(v)

            # ========== Bicubic ==========
            start_time = time.time()
            sr_bicubic = bicubic(lr_img)
            inference_times['bicubic'].append(time.time() - start_time)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            for k, v in metrics_bicubic.items():
                results['bicubic'][k].append(v)

            # ========== SRCNN ==========
            start_time = time.time()
            sr_srcnn = srcnn(lr_img)
            inference_times['srcnn'].append(time.time() - start_time)
            metrics_srcnn = sr_metrics.evaluate_batch(sr_srcnn, hr_img)
            for k, v in metrics_srcnn.items():
                results['srcnn'][k].append(v)

            # ========== RFB-ESRGAN (Ours) ==========
            start_time = time.time()
            sr_ours = generator(lr_img)
            inference_times['ours'].append(time.time() - start_time)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            for k, v in metrics_ours.items():
                results['ours'][k].append(v)

            sample_count += lr_img.size(0)

    # ========== Calculate Average Metrics ==========
    print("\n" + "="*70)
    print("RESULTS SUMMARY")
    print("="*70)

    comparison_table = []

    for model_name in model_names:
        avg_metrics = {k: np.mean(v) for k, v in results[model_name].items()}
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms

        print(f"\n{model_name.upper()}:")
        print(f"  PSNR: {avg_metrics['psnr']:.2f} dB")
        print(f"  SSIM: {avg_metrics['ssim']:.4f}")
        print(f"  MS-SSIM: {avg_metrics['ms_ssim']:.4f}")
        print(f"  LPIPS: {avg_metrics['lpips']:.4f} (lower is better)")
        print(f"  MAE: {avg_metrics['mae']:.4f}")
        print(f"  RMSE: {avg_metrics['rmse']:.4f}")
        print(f"  NDVI Error: {avg_metrics['ndvi_error']:.4f}")
        print(f"  Edge Preservation: {avg_metrics['edge_preservation']:.4f}")
        print(f"  Inference Time: {avg_time:.2f} ms/image")

        comparison_table.append({
            'model': model_name,
            **avg_metrics,
            'inference_time_ms': avg_time
        })

    # ========== Calculate Improvement Deltas ==========
    print("\n" + "="*70)
    print("IMPROVEMENT vs. BASELINES")
    print("="*70)

    ours_psnr = np.mean(results['ours']['psnr'])
    ours_ssim = np.mean(results['ours']['ssim'])
    bicubic_psnr = np.mean(results['bicubic']['psnr'])
    bicubic_ssim = np.mean(results['bicubic']['ssim'])
    srcnn_psnr = np.mean(results['srcnn']['psnr'])
    srcnn_ssim = np.mean(results['srcnn']['ssim'])

    delta_psnr_bicubic = ours_psnr - bicubic_psnr
    delta_ssim_bicubic = ours_ssim - bicubic_ssim
    delta_psnr_srcnn = ours_psnr - srcnn_psnr
    delta_ssim_srcnn = ours_ssim - srcnn_ssim

    print(f"\nΔPSNR vs. Bicubic: +{delta_psnr_bicubic:.2f} dB ({delta_psnr_bicubic/bicubic_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. Bicubic: +{delta_ssim_bicubic:.4f} ({delta_ssim_bicubic/bicubic_ssim*100:.1f}% improvement)")
    print(f"ΔPSNR vs. SRCNN: +{delta_psnr_srcnn:.2f} dB ({delta_psnr_srcnn/srcnn_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. SRCNN: +{delta_ssim_srcnn:.4f} ({delta_ssim_srcnn/srcnn_ssim*100:.1f}% improvement)")

    # ========== Model Efficiency ==========
    print("\n" + "="*70)
    print("MODEL EFFICIENCY METRICS")
    print("="*70)

    # Parameter count
    def count_parameters(model):
        if isinstance(model, nn.DataParallel):
            return sum(p.numel() for p in model.module.parameters())
        return sum(p.numel() for p in model.parameters())

    ours_params = count_parameters(generator)
    srcnn_params = count_parameters(srcnn)

    print(f"\nParameter Count:")
    print(f"  RFB-ESRGAN (Ours): {ours_params/1e6:.2f}M parameters")
    print(f"  SRCNN: {srcnn_params/1e6:.2f}M parameters")

    # Parameter efficiency
    psnr_per_param_ours = (ours_psnr - bicubic_psnr) / (ours_params / 1e6)
    psnr_per_param_srcnn = (srcnn_psnr - bicubic_psnr) / (srcnn_params / 1e6)

    print(f"\nParameter Efficiency (ΔPSNR per 1M params vs. Bicubic):")
    print(f"  RFB-ESRGAN: {psnr_per_param_ours:.3f} dB/M")
    print(f"  SRCNN: {psnr_per_param_srcnn:.3f} dB/M")

    # ========== System Performance Metrics ==========
    print("\n" + "="*70)
    print("SYSTEM PERFORMANCE METRICS")
    print("="*70)

    avg_time_ours = np.mean(inference_times['ours'])
    fps_ours = 1.0 / avg_time_ours if avg_time_ours > 0 else 0

    print(f"\nInference Performance (Ours):")
    print(f"  Latency: {avg_time_ours*1000:.2f} ms/image")
    print(f"  Throughput: {fps_ours:.2f} FPS")
    print(f"  Speed vs. Bicubic: {np.mean(inference_times['bicubic'])/avg_time_ours:.2f}x slower")

    # GPU Memory footprint
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = generator(lr_img)
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**2  # MB
        print(f"  GPU Memory Footprint: {memory_allocated:.2f} MB")

    # ========== Statistical Significance ==========
    print("\n" + "="*70)
    print("STATISTICAL ANALYSIS")
    print("="*70)

    from scipy import stats

    # T-test comparing our model vs bicubic
    t_stat, p_value = stats.ttest_rel(results['ours']['psnr'], results['bicubic']['psnr'])
    print(f"\nPSNR T-test (Ours vs. Bicubic):")
    print(f"  t-statistic: {t_stat:.3f}")
    print(f"  p-value: {p_value:.6f}")
    print(f"  Statistically significant: {'Yes' if p_value < 0.05 else 'No'} (p < 0.05)")

    # Standard deviations
    print(f"\nStandard Deviations:")
    for model_name in ['bicubic', 'srcnn', 'ours']:
        std_psnr = np.std(results[model_name]['psnr'])
        std_ssim = np.std(results[model_name]['ssim'])
        print(f"  {model_name.upper()}: PSNR±{std_psnr:.2f}, SSIM±{std_ssim:.4f}")

    # ========== Log to WandB (with error handling) ==========
    try:
        if wandb.run is not None:
            wandb.log({
                'eval/psnr_ours': ours_psnr,
                'eval/ssim_ours': ours_ssim,
                'eval/ms_ssim_ours': np.mean(results['ours']['ms_ssim']),
                'eval/lpips_ours': np.mean(results['ours']['lpips']),
                'eval/delta_psnr_vs_bicubic': delta_psnr_bicubic,
                'eval/delta_ssim_vs_bicubic': delta_ssim_bicubic,
                'eval/inference_time_ms': avg_time_ours * 1000,
                'eval/throughput_fps': fps_ours,
                'eval/parameters_millions': ours_params / 1e6,
            })
    except Exception as e:
        print(f"\n⚠️  Warning: Could not log to WandB: {e}")
        print("   (This is not critical - evaluation results are still available)")

    return comparison_table, results, inference_times


print("✓ Comparative evaluation function defined!")

In [ ]:
# ========== 4. VISUALIZATION FUNCTIONS ==========

def create_comparison_visualizations(results, inference_times, save_dir):
    """Create comprehensive comparison plots"""
    os.makedirs(save_dir, exist_ok=True)
    
    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    
    # Set style
    sns.set_style("whitegrid")
    colors = plt.cm.Set2(range(5))
    
    # ========== Figure 1: Main Metrics Bar Chart ==========
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PSNR
    psnr_values = [np.mean(results[m]['psnr']) for m in model_names]
    axes[0, 0].bar(model_names, psnr_values, color=colors)
    axes[0, 0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0, 0].set_title('Peak Signal-to-Noise Ratio', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # SSIM
    ssim_values = [np.mean(results[m]['ssim']) for m in model_names]
    axes[0, 1].bar(model_names, ssim_values, color=colors)
    axes[0, 1].set_ylabel('SSIM', fontsize=11)
    axes[0, 1].set_title('Structural Similarity Index', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # MS-SSIM
    msssim_values = [np.mean(results[m]['ms_ssim']) for m in model_names]
    axes[1, 0].bar(model_names, msssim_values, color=colors)
    axes[1, 0].set_ylabel('MS-SSIM', fontsize=11)
    axes[1, 0].set_title('Multi-Scale SSIM', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # LPIPS
    lpips_values = [np.mean(results[m]['lpips']) for m in model_names]
    axes[1, 1].bar(model_names, lpips_values, color=colors)
    axes[1, 1].set_ylabel('LPIPS (lower is better)', fontsize=11)
    axes[1, 1].set_title('Perceptual Distance (LPIPS)', fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'metrics_comparison.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 2: Box Plots for Metric Distributions ==========
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # PSNR distribution
    psnr_data = [results[m]['psnr'] for m in model_names]
    bp1 = axes[0].boxplot(psnr_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp1['boxes'], colors):
        patch.set_facecolor(color)
    axes[0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0].set_title('PSNR Distribution', fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # SSIM distribution
    ssim_data = [results[m]['ssim'] for m in model_names]
    bp2 = axes[1].boxplot(ssim_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp2['boxes'], colors):
        patch.set_facecolor(color)
    axes[1].set_ylabel('SSIM', fontsize=11)
    axes[1].set_title('SSIM Distribution', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    # LPIPS distribution
    lpips_data = [results[m]['lpips'] for m in model_names]
    bp3 = axes[2].boxplot(lpips_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp3['boxes'], colors):
        patch.set_facecolor(color)
    axes[2].set_ylabel('LPIPS', fontsize=11)
    axes[2].set_title('LPIPS Distribution (lower is better)', fontweight='bold')
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'metrics_distribution.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 3: Radar Chart ==========
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    
    # Normalize metrics to 0-1 for radar chart
    categories = ['PSNR', 'SSIM', 'MS-SSIM', 'Edge\nPreserv.', 'Speed']
    
    for i, model_name in enumerate(['bicubic', 'srcnn', 'ours']):
        values = [
            np.mean(results[model_name]['psnr']) / 35.0,  # Normalize to ~35 dB max
            np.mean(results[model_name]['ssim']),
            np.mean(results[model_name]['ms_ssim']),
            np.mean(results[model_name]['edge_preservation']),
            1.0 / (np.mean(inference_times[model_name]) * 100)  # Inverse time
        ]
        
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
        values += values[:1]
        angles += angles[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=model_name.upper(), color=colors[model_names.index(model_name)])
        ax.fill(angles, values, alpha=0.15, color=colors[model_names.index(model_name)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title('Multi-Metric Performance Comparison', fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True)
    
    fig_path = os.path.join(save_dir, 'radar_chart.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 4: Quality-Speed Tradeoff ==========
    plt.figure(figsize=(10, 6))
    
    for i, model_name in enumerate(model_names):
        avg_psnr = np.mean(results[model_name]['psnr'])
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms
        
        plt.scatter(avg_time, avg_psnr, s=200, color=colors[i], label=model_name.upper(), alpha=0.7, edgecolors='black')
        plt.annotate(model_name.upper(), (avg_time, avg_psnr), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    plt.xlabel('Inference Time (ms/image)', fontsize=11)
    plt.ylabel('PSNR (dB)', fontsize=11)
    plt.title('Quality-Speed Tradeoff', fontsize=13, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    
    fig_path = os.path.join(save_dir, 'quality_speed_tradeoff.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    print(f"\n✓ All comparison visualizations saved to: {save_dir}")


print("✓ Visualization functions defined!")

In [ ]:
# ========== 5. VISUAL QUALITY COMPARISON ==========

def visualize_quality_comparison(generator, val_loader, device, save_dir, num_samples=5):
    """Generate side-by-side visual comparisons"""
    os.makedirs(save_dir, exist_ok=True)
    
    bicubic = BicubicUpsampler(scale_factor=8)
    sr_metrics = SuperResolutionMetrics(device)
    
    generator.eval()
    sample_idx = 0
    
    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            if sample_idx >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR images
            sr_bicubic = bicubic(lr_img)
            sr_ours = generator(lr_img)
            
            # Calculate metrics
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            
            # Visualize first image in batch
            fig, axes = plt.subplots(1, 4, figsize=(18, 5))
            
            # LR (upsampled for visualization)
            lr_upsampled = F.interpolate(lr_img, scale_factor=8, mode='nearest')
            lr_np = lr_upsampled[0].cpu().permute(1, 2, 0).numpy()
            lr_np = np.clip(lr_np, 0, 1)
            axes[0].imshow(lr_np)
            axes[0].set_title(f'LR Input\n(×8 nearest)', fontsize=11, fontweight='bold')
            axes[0].axis('off')
            
            # HR (Ground Truth)
            hr_np = hr_img[0].cpu().permute(1, 2, 0).numpy()
            hr_np = np.clip(hr_np, 0, 1)
            axes[1].imshow(hr_np)
            axes[1].set_title('HR Ground Truth', fontsize=11, fontweight='bold')
            axes[1].axis('off')
            
            # Bicubic
            bicubic_np = sr_bicubic[0].cpu().permute(1, 2, 0).numpy()
            bicubic_np = np.clip(bicubic_np, 0, 1)
            axes[2].imshow(bicubic_np)
            axes[2].set_title(f'Bicubic\nPSNR: {metrics_bicubic["psnr"]:.2f} dB\nSSIM: {metrics_bicubic["ssim"]:.3f}', 
                            fontsize=10, fontweight='bold')
            axes[2].axis('off')
            
            # Ours
            ours_np = sr_ours[0].cpu().permute(1, 2, 0).numpy()
            ours_np = np.clip(ours_np, 0, 1)
            axes[3].imshow(ours_np)
            axes[3].set_title(f'RFB-ESRGAN (Ours)\nPSNR: {metrics_ours["psnr"]:.2f} dB\nSSIM: {metrics_ours["ssim"]:.3f}', 
                            fontsize=10, fontweight='bold', color='green')
            axes[3].axis('off')
            
            plt.tight_layout()
            fig_path = os.path.join(save_dir, f'comparison_sample_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            print(f"✓ Saved: comparison_sample_{sample_idx+1}.png")
            plt.close()
            
            sample_idx += 1
    
    print(f"\n✓ Quality comparison visualizations saved to: {save_dir}")


print("✓ Visual quality comparison function defined!")

In [ ]:
# ========== 6. DIFFERENCE MAPS VISUALIZATION ==========

def visualize_difference_maps(generator, val_loader, device, save_dir, num_samples=5):
    """Visualize pixel-wise error maps"""
    os.makedirs(save_dir, exist_ok=True)
    
    bicubic = BicubicUpsampler(scale_factor=8)
    
    generator.eval()
    sample_idx = 0
    
    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            if sample_idx >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR images
            sr_bicubic = bicubic(lr_img)
            sr_ours = generator(lr_img)
            
            # Calculate absolute error maps
            error_bicubic = torch.abs(sr_bicubic - hr_img).mean(dim=1, keepdim=True)  # Average across RGB
            error_ours = torch.abs(sr_ours - hr_img).mean(dim=1, keepdim=True)
            
            # Visualize
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            
            # Row 1: Ground Truth, Bicubic SR, Ours SR
            hr_np = hr_img[0].cpu().permute(1, 2, 0).numpy()
            hr_np = np.clip(hr_np, 0, 1)
            axes[0, 0].imshow(hr_np)
            axes[0, 0].set_title('Ground Truth (HR)', fontsize=11, fontweight='bold')
            axes[0, 0].axis('off')
            
            bicubic_np = sr_bicubic[0].cpu().permute(1, 2, 0).numpy()
            bicubic_np = np.clip(bicubic_np, 0, 1)
            axes[0, 1].imshow(bicubic_np)
            axes[0, 1].set_title('Bicubic SR', fontsize=11, fontweight='bold')
            axes[0, 1].axis('off')
            
            ours_np = sr_ours[0].cpu().permute(1, 2, 0).numpy()
            ours_np = np.clip(ours_np, 0, 1)
            axes[0, 2].imshow(ours_np)
            axes[0, 2].set_title('RFB-ESRGAN SR (Ours)', fontsize=11, fontweight='bold')
            axes[0, 2].axis('off')
            
            # Row 2: Error maps
            axes[1, 0].axis('off')  # Empty cell
            
            error_bicubic_np = error_bicubic[0, 0].cpu().numpy()
            mae_bicubic = error_bicubic_np.mean()
            im1 = axes[1, 1].imshow(error_bicubic_np, cmap='hot', vmin=0, vmax=0.3)
            axes[1, 1].set_title(f'Bicubic Error Map\nMAE: {mae_bicubic:.4f}', fontsize=11, fontweight='bold')
            axes[1, 1].axis('off')
            plt.colorbar(im1, ax=axes[1, 1], fraction=0.046, pad=0.04)
            
            error_ours_np = error_ours[0, 0].cpu().numpy()
            mae_ours = error_ours_np.mean()
            im2 = axes[1, 2].imshow(error_ours_np, cmap='hot', vmin=0, vmax=0.3)
            axes[1, 2].set_title(f'RFB-ESRGAN Error Map\nMAE: {mae_ours:.4f}', fontsize=11, fontweight='bold', color='green')
            axes[1, 2].axis('off')
            plt.colorbar(im2, ax=axes[1, 2], fraction=0.046, pad=0.04)
            
            plt.tight_layout()
            fig_path = os.path.join(save_dir, f'error_map_sample_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            print(f"✓ Saved: error_map_sample_{sample_idx+1}.png")
            plt.close()
            
            sample_idx += 1
    
    print(f"\n✓ Difference maps saved to: {save_dir}")


print("✓ Difference maps visualization function defined!")

In [ ]:
# ========== 6. ENHANCED AGRICULTURAL-SPECIFIC VISUALIZATIONS ==========

def visualize_agricultural_predictions_enhanced(generator, val_loader, device, save_dir, num_samples=5):
    """Enhanced visualization suite for agricultural super-resolution"""
    os.makedirs(save_dir, exist_ok=True)
    
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    
    generator.eval()
    
    sample_idx = 0
    
    print("\n🌾 Generating Enhanced Agricultural Visualizations...")
    
    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            if sample_idx >= num_samples:
                break
                
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate predictions
            sr_ours = generator(lr_img)
            sr_bicubic = bicubic(lr_img)
            
            # Denormalize all images
            def denorm(t):
                return torch.clamp((t * 0.5 + 0.5) * 255, 0, 255).byte()
            
            lr_np = denorm(lr_img[0]).cpu().permute(1, 2, 0).numpy()
            hr_np = denorm(hr_img[0]).cpu().permute(1, 2, 0).numpy()
            sr_np = denorm(sr_ours[0]).cpu().permute(1, 2, 0).numpy()
            bicubic_np = denorm(sr_bicubic[0]).cpu().permute(1, 2, 0).numpy()
            
            # ========== FIGURE 1: COMPREHENSIVE COMPARISON ==========
            fig = plt.figure(figsize=(20, 12))
            gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
            
            # Row 1: Full Image Comparisons
            ax1 = fig.add_subplot(gs[0, 0])
            ax1.imshow(lr_np)
            ax1.set_title('Low Resolution Input\n(32×32)', fontsize=12, fontweight='bold')
            ax1.axis('off')
            
            ax2 = fig.add_subplot(gs[0, 1])
            ax2.imshow(bicubic_np)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic[:1], hr_img[:1])
            ax2.set_title(f'Bicubic Baseline\nPSNR: {metrics_bicubic["psnr"]:.2f} | SSIM: {metrics_bicubic["ssim"]:.4f}', 
                         fontsize=11)
            ax2.axis('off')
            
            ax3 = fig.add_subplot(gs[0, 2])
            ax3.imshow(sr_np)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours[:1], hr_img[:1])
            ax3.set_title(f'Our SR Model\nPSNR: {metrics_ours["psnr"]:.2f} | SSIM: {metrics_ours["ssim"]:.4f}', 
                         fontsize=11, color='green', fontweight='bold')
            ax3.axis('off')
            
            ax4 = fig.add_subplot(gs[0, 3])
            ax4.imshow(hr_np)
            ax4.set_title('Ground Truth\n(256×256)', fontsize=12, fontweight='bold')
            ax4.axis('off')
            
            # Row 2: Detailed Crops
            crop_size = 64
            start_y, start_x = 96, 96
            
            for idx, (img, title) in enumerate([
                (lr_np, 'LR Crop'),
                (bicubic_np, 'Bicubic Crop'),
                (sr_np, 'Our SR Crop'),
                (hr_np, 'GT Crop')
            ]):
                ax = fig.add_subplot(gs[1, idx])
                if idx == 0:
                    crop = img[start_y//8:(start_y+crop_size)//8, start_x//8:(start_x+crop_size)//8]
                else:
                    crop = img[start_y:start_y+crop_size, start_x:start_x+crop_size]
                ax.imshow(crop)
                ax.set_title(title, fontsize=11)
                ax.axis('off')
                rect = plt.Rectangle((0, 0), crop.shape[1]-1, crop.shape[0]-1, 
                                    fill=False, color='red', linewidth=2)
                ax.add_patch(rect)
            
            # Row 3: Difference Maps & Analysis
            diff_ours = np.abs(sr_np.astype(float) - hr_np.astype(float)).mean(axis=2)
            diff_bicubic = np.abs(bicubic_np.astype(float) - hr_np.astype(float)).mean(axis=2)
            
            ax = fig.add_subplot(gs[2, 0])
            im = ax.imshow(diff_bicubic, cmap='hot', vmin=0, vmax=50)
            ax.set_title(f'Bicubic Error Map\nMean Error: {diff_bicubic.mean():.2f}', fontsize=11)
            ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)
            
            ax = fig.add_subplot(gs[2, 1])
            im = ax.imshow(diff_ours, cmap='hot', vmin=0, vmax=50)
            ax.set_title(f'Our Model Error Map\nMean Error: {diff_ours.mean():.2f}', fontsize=11)
            ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)
            
            # Histogram of errors
            ax = fig.add_subplot(gs[2, 2])
            ax.hist(diff_bicubic.ravel(), bins=50, alpha=0.5, color='orange', label='Bicubic', density=True)
            ax.hist(diff_ours.ravel(), bins=50, alpha=0.5, color='green', label='Our Model', density=True)
            ax.set_xlabel('Pixel Error', fontsize=10)
            ax.set_ylabel('Density', fontsize=10)
            ax.set_title('Error Distribution', fontsize=11, fontweight='bold')
            ax.legend()
            ax.grid(alpha=0.3)
            
            # Metrics comparison bar chart
            ax = fig.add_subplot(gs[2, 3])
            metrics = ['PSNR', 'SSIM']
            bicubic_vals = [metrics_bicubic['psnr']/30, metrics_bicubic['ssim']]
            ours_vals = [metrics_ours['psnr']/30, metrics_ours['ssim']]
            x = np.arange(len(metrics))
            width = 0.35
            ax.bar(x - width/2, bicubic_vals, width, label='Bicubic', color='orange', alpha=0.7)
            ax.bar(x + width/2, ours_vals, width, label='Our Model', color='green', alpha=0.7)
            ax.set_ylabel('Normalized Score', fontsize=10)
            ax.set_title('Metrics Comparison', fontsize=11, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(metrics)
            ax.legend()
            ax.grid(axis='y', alpha=0.3)
            
            plt.suptitle(f'Sample {sample_idx + 1}: Agricultural Super-Resolution Analysis', 
                        fontsize=14, fontweight='bold', y=0.98)
            
            fig_path = os.path.join(save_dir, f'enhanced_comparison_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ Saved: enhanced_comparison_{sample_idx+1}.png")
            
            # ========== FIGURE 2: SPECTRAL & NDVI ANALYSIS ==========
            # Calculate NDVI-like features from RGB
            # For RGB images, we approximate: NDVI ≈ (G - R) / (G + R + 0.1)
            def calculate_pseudo_ndvi(rgb_img):
                rgb_float = rgb_img.astype(float)
                r, g = rgb_float[:,:,0], rgb_float[:,:,1]
                pseudo_ndvi = (g - r) / (g + r + 0.1)
                return pseudo_ndvi
            
            ndvi_lr = calculate_pseudo_ndvi(lr_np)
            ndvi_bicubic = calculate_pseudo_ndvi(bicubic_np)
            ndvi_ours = calculate_pseudo_ndvi(sr_np)
            ndvi_gt = calculate_pseudo_ndvi(hr_np)
            
            fig, axes = plt.subplots(2, 4, figsize=(20, 10))
            
            # Row 1: NDVI Maps
            for idx, (ndvi, title) in enumerate([
                (ndvi_lr, 'LR NDVI'),
                (ndvi_bicubic, 'Bicubic NDVI'),
                (ndvi_ours, 'Our SR NDVI'),
                (ndvi_gt, 'GT NDVI')
            ]):
                im = axes[0, idx].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
                axes[0, idx].set_title(title, fontsize=12, fontweight='bold')
                axes[0, idx].axis('off')
                plt.colorbar(im, ax=axes[0, idx], fraction=0.046)
            
            # Row 2: Spectral Analysis
            # RGB channel histograms
            for idx, (img, title, color_map) in enumerate([
                (lr_np, 'LR Spectrum', ['darkred', 'darkgreen', 'darkblue']),
                (bicubic_np, 'Bicubic Spectrum', ['salmon', 'lightgreen', 'lightblue']),
                (sr_np, 'Our SR Spectrum', ['red', 'green', 'blue']),
                (hr_np, 'GT Spectrum', ['maroon', 'darkgreen', 'navy'])
            ]):
                for c, color in enumerate(color_map):
                    axes[1, idx].hist(img[:,:,c].ravel(), bins=50, alpha=0.5, 
                                     color=color, label=['R','G','B'][c], density=True)
                axes[1, idx].set_xlabel('Pixel Intensity', fontsize=10)
                axes[1, idx].set_ylabel('Density', fontsize=10)
                axes[1, idx].set_title(title, fontsize=11, fontweight='bold')
                axes[1, idx].legend()
                axes[1, idx].grid(alpha=0.3)
            
            plt.suptitle(f'Sample {sample_idx + 1}: Agricultural Spectral & NDVI Analysis', 
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            
            fig_path = os.path.join(save_dir, f'spectral_analysis_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ Saved: spectral_analysis_{sample_idx+1}.png")
            
            # ========== FIGURE 3: TEXTURE & EDGE ANALYSIS ==========
            from scipy.ndimage import sobel
            
            def calculate_edges(img):
                gray = np.mean(img, axis=2)
                edge_x = sobel(gray, axis=0)
                edge_y = sobel(gray, axis=1)
                edges = np.hypot(edge_x, edge_y)
                return edges
            
            edges_bicubic = calculate_edges(bicubic_np)
            edges_ours = calculate_edges(sr_np)
            edges_gt = calculate_edges(hr_np)
            
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            
            # Row 1: Edge Maps
            for idx, (edges, title) in enumerate([
                (edges_bicubic, 'Bicubic Edges'),
                (edges_ours, 'Our SR Edges'),
                (edges_gt, 'GT Edges')
            ]):
                im = axes[0, idx].imshow(edges, cmap='viridis')
                axes[0, idx].set_title(title, fontsize=12, fontweight='bold')
                axes[0, idx].axis('off')
                plt.colorbar(im, ax=axes[0, idx], fraction=0.046)
            
            # Row 2: Texture Analysis (Local Standard Deviation)
            from scipy.ndimage import generic_filter
            
            def local_std(img):
                gray = np.mean(img, axis=2)
                return generic_filter(gray, np.std, size=5)
            
            texture_bicubic = local_std(bicubic_np)
            texture_ours = local_std(sr_np)
            texture_gt = local_std(hr_np)
            
            for idx, (texture, title) in enumerate([
                (texture_bicubic, 'Bicubic Texture'),
                (texture_ours, 'Our SR Texture'),
                (texture_gt, 'GT Texture')
            ]):
                im = axes[1, idx].imshow(texture, cmap='plasma')
                axes[1, idx].set_title(title, fontsize=12, fontweight='bold')
                axes[1, idx].axis('off')
                plt.colorbar(im, ax=axes[1, idx], fraction=0.046)
            
            plt.suptitle(f'Sample {sample_idx + 1}: Edge & Texture Detail Analysis', 
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            
            fig_path = os.path.join(save_dir, f'texture_analysis_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ Saved: texture_analysis_{sample_idx+1}.png")
            
            # ========== SAVE NUMERICAL METRICS ==========
            metrics_dict = {
                'sample_id': sample_idx + 1,
                'psnr_bicubic': float(metrics_bicubic['psnr']),
                'ssim_bicubic': float(metrics_bicubic['ssim']),
                'psnr_ours': float(metrics_ours['psnr']),
                'ssim_ours': float(metrics_ours['ssim']),
                'psnr_improvement': float(metrics_ours['psnr'] - metrics_bicubic['psnr']),
                'ssim_improvement': float(metrics_ours['ssim'] - metrics_bicubic['ssim']),
                'mean_error_bicubic': float(diff_bicubic.mean()),
                'mean_error_ours': float(diff_ours.mean()),
                'ndvi_correlation_bicubic': float(np.corrcoef(ndvi_bicubic.ravel(), ndvi_gt.ravel())[0,1]),
                'ndvi_correlation_ours': float(np.corrcoef(ndvi_ours.ravel(), ndvi_gt.ravel())[0,1]),
                'edge_similarity_bicubic': float(np.corrcoef(edges_bicubic.ravel(), edges_gt.ravel())[0,1]),
                'edge_similarity_ours': float(np.corrcoef(edges_ours.ravel(), edges_gt.ravel())[0,1])
            }
            
            metrics_path = os.path.join(save_dir, f'metrics_{sample_idx+1}.json')
            with open(metrics_path, 'w') as f:
                json.dump(metrics_dict, f, indent=2)
            print(f"✓ Saved: metrics_{sample_idx+1}.json")
            
            # Print summary
            print(f"\n{'='*60}")
            print(f"Sample {sample_idx+1} Analysis Summary:")
            print(f"  PSNR: {metrics_ours['psnr']:.2f} dB (Δ+{metrics_ours['psnr']-metrics_bicubic['psnr']:.2f})")
            print(f"  SSIM: {metrics_ours['ssim']:.4f} (Δ+{metrics_ours['ssim']-metrics_bicubic['ssim']:.4f})")
            print(f"  NDVI Corr: {metrics_dict['ndvi_correlation_ours']:.4f} (Bicubic: {metrics_dict['ndvi_correlation_bicubic']:.4f})")
            print(f"  Edge Sim: {metrics_dict['edge_similarity_ours']:.4f} (Bicubic: {metrics_dict['edge_similarity_bicubic']:.4f})")
            print(f"{'='*60}\n")
            
            sample_idx += 1
    
    print(f"\n✓ NDVI spectral analysis saved to: {save_dir}")
    generator.train()


print("✓ Enhanced agricultural-specific visualization functions defined!")

In [ ]:
# ========== 7. MODEL CONVERGENCE ANALYSIS ==========

def analyze_model_convergence(generator, val_loader, device, save_dir):
    """Analyze prediction quality distribution and stability"""
    os.makedirs(save_dir, exist_ok=True)
    
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    
    psnr_list = []
    ssim_list = []
    improvements_psnr = []
    improvements_ssim = []
    
    generator.eval()
    
    print("\n📊 Analyzing model convergence and stability...")
    
    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Analyzing", ncols=80):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Our model
            sr_ours = generator(lr_img)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            
            # Bicubic baseline
            sr_bicubic = bicubic(lr_img)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            
            # Store metrics
            psnr_list.append(metrics_ours['psnr'])
            ssim_list.append(metrics_ours['ssim'])
            
            # Store improvements
            improvements_psnr.append(metrics_ours['psnr'] - metrics_bicubic['psnr'])
            improvements_ssim.append(metrics_ours['ssim'] - metrics_bicubic['ssim'])
    
    # ========== Figure 1: Metric Distributions ==========
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PSNR histogram
    axes[0, 0].hist(psnr_list, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(np.mean(psnr_list), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(psnr_list):.2f} dB')
    axes[0, 0].set_xlabel('PSNR (dB)', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('PSNR Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # SSIM histogram
    axes[0, 1].hist(ssim_list, bins=50, color='seagreen', alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(np.mean(ssim_list), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(ssim_list):.4f}')
    axes[0, 1].set_xlabel('SSIM', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('SSIM Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # PSNR improvement distribution
    axes[1, 0].hist(improvements_psnr, bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(np.mean(improvements_psnr), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: +{np.mean(improvements_psnr):.2f} dB')
    axes[1, 0].set_xlabel('ΔPSNR vs. Bicubic (dB)', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('PSNR Improvement Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # SSIM improvement distribution
    axes[1, 1].hist(improvements_ssim, bins=50, color='coral', alpha=0.7, edgecolor='black')
    axes[1, 1].axvline(np.mean(improvements_ssim), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: +{np.mean(improvements_ssim):.4f}')
    axes[1, 1].set_xlabel('ΔSSIM vs. Bicubic', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title('SSIM Improvement Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'convergence_analysis.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: convergence_analysis.png")
    plt.close()
    
    # ========== Figure 2: Performance Trends ==========
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # PSNR trend
    axes[0].plot(psnr_list, alpha=0.6, linewidth=0.8, color='steelblue')
    axes[0].axhline(np.mean(psnr_list), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[0].fill_between(range(len(psnr_list)), 
                         np.mean(psnr_list) - np.std(psnr_list), 
                         np.mean(psnr_list) + np.std(psnr_list), 
                         alpha=0.2, color='red', label='±1 Std Dev')
    axes[0].set_xlabel('Sample Index', fontsize=11)
    axes[0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0].set_title('PSNR Across Validation Set', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # SSIM trend
    axes[1].plot(ssim_list, alpha=0.6, linewidth=0.8, color='seagreen')
    axes[1].axhline(np.mean(ssim_list), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[1].fill_between(range(len(ssim_list)), 
                         np.mean(ssim_list) - np.std(ssim_list), 
                         np.mean(ssim_list) + np.std(ssim_list), 
                         alpha=0.2, color='red', label='±1 Std Dev')
    axes[1].set_xlabel('Sample Index', fontsize=11)
    axes[1].set_ylabel('SSIM', fontsize=11)
    axes[1].set_title('SSIM Across Validation Set', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'performance_trends.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: performance_trends.png")
    plt.close()
    
    # ========== Statistics Summary ==========
    print("\n" + "="*70)
    print("CONVERGENCE ANALYSIS SUMMARY")
    print("="*70)
    print(f"\nPSNR Statistics:")
    print(f"  Mean: {np.mean(psnr_list):.2f} dB")
    print(f"  Std Dev: {np.std(psnr_list):.2f} dB")
    print(f"  Min: {np.min(psnr_list):.2f} dB")
    print(f"  Max: {np.max(psnr_list):.2f} dB")
    print(f"  Median: {np.median(psnr_list):.2f} dB")
    
    print(f"\nSSIM Statistics:")
    print(f"  Mean: {np.mean(ssim_list):.4f}")
    print(f"  Std Dev: {np.std(ssim_list):.4f}")
    print(f"  Min: {np.min(ssim_list):.4f}")
    print(f"  Max: {np.max(ssim_list):.4f}")
    print(f"  Median: {np.median(ssim_list):.4f}")
    
    print(f"\nImprovement Statistics (vs. Bicubic):")
    print(f"  ΔPSNR Mean: +{np.mean(improvements_psnr):.2f} dB")
    print(f"  ΔPSNR Std Dev: {np.std(improvements_psnr):.2f} dB")
    print(f"  ΔSSIM Mean: +{np.mean(improvements_ssim):.4f}")
    print(f"  ΔSSIM Std Dev: {np.std(improvements_ssim):.4f}")
    
    # Coefficient of Variation (measure of stability)
    cv_psnr = (np.std(psnr_list) / np.mean(psnr_list)) * 100
    cv_ssim = (np.std(ssim_list) / np.mean(ssim_list)) * 100
    
    print(f"\nStability Metrics (Coefficient of Variation):")
    print(f"  PSNR CV: {cv_psnr:.2f}% (lower is more stable)")
    print(f"  SSIM CV: {cv_ssim:.2f}% (lower is more stable)")
    
    print(f"\n✓ Convergence analysis completed and saved to: {save_dir}")


print("✓ Convergence analysis function defined!")

In [ ]:
# ========== 8. FAILURE CASE ANALYSIS ==========

def analyze_failure_cases(generator, val_loader, device, save_dir, num_worst=10):
    """Identify and visualize worst-performing samples"""
    os.makedirs(save_dir, exist_ok=True)
    
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    
    all_samples = []
    
    generator.eval()
    
    print("\n🔍 Identifying failure cases...")
    
    with torch.no_grad():
        for batch_idx, (lr_img, hr_img) in enumerate(tqdm(val_loader, desc="Scanning", ncols=80)):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR
            sr_ours = generator(lr_img)
            sr_bicubic = bicubic(lr_img)
            
            # Calculate metrics
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            
            # Store sample info
            all_samples.append({
                'batch_idx': batch_idx,
                'lr': lr_img[0].cpu(),
                'hr': hr_img[0].cpu(),
                'sr_ours': sr_ours[0].cpu(),
                'sr_bicubic': sr_bicubic[0].cpu(),
                'psnr': metrics_ours['psnr'],
                'ssim': metrics_ours['ssim'],
                'improvement_psnr': metrics_ours['psnr'] - metrics_bicubic['psnr'],
                'improvement_ssim': metrics_ours['ssim'] - metrics_bicubic['ssim'],
            })
    
    # ========== Find Worst Cases ==========
    # Sort by PSNR (ascending)
    worst_psnr = sorted(all_samples, key=lambda x: x['psnr'])[:num_worst]
    
    # Sort by improvement over bicubic (ascending - least improvement or regression)
    worst_improvement = sorted(all_samples, key=lambda x: x['improvement_psnr'])[:num_worst]
    
    print(f"\n🔴 Found {num_worst} worst-performing samples")
    
    # ========== Visualize Worst PSNR Cases ==========
    fig, axes = plt.subplots(num_worst, 4, figsize=(16, 4*num_worst))
    
    if num_worst == 1:
        axes = axes.reshape(1, -1)
    
    for i, sample in enumerate(worst_psnr):
        # LR
        lr_np = F.interpolate(sample['lr'].unsqueeze(0), scale_factor=8, mode='nearest')[0].permute(1, 2, 0).numpy()
        lr_np = np.clip(lr_np, 0, 1)
        axes[i, 0].imshow(lr_np)
        axes[i, 0].set_title(f'LR Input (Sample #{sample["batch_idx"]})', fontsize=9)
        axes[i, 0].axis('off')
        
        # HR
        hr_np = sample['hr'].permute(1, 2, 0).numpy()
        hr_np = np.clip(hr_np, 0, 1)
        axes[i, 1].imshow(hr_np)
        axes[i, 1].set_title('HR Ground Truth', fontsize=9)
        axes[i, 1].axis('off')
        
        # Bicubic
        bicubic_np = sample['sr_bicubic'].permute(1, 2, 0).numpy()
        bicubic_np = np.clip(bicubic_np, 0, 1)
        axes[i, 2].imshow(bicubic_np)
        axes[i, 2].set_title('Bicubic SR', fontsize=9)
        axes[i, 2].axis('off')
        
        # Ours
        ours_np = sample['sr_ours'].permute(1, 2, 0).numpy()
        ours_np = np.clip(ours_np, 0, 1)
        axes[i, 3].imshow(ours_np)
        axes[i, 3].set_title(f'Ours (PSNR: {sample["psnr"]:.2f} dB)', fontsize=9, color='red')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'Top {num_worst} Worst PSNR Cases', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'worst_psnr_cases.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: worst_psnr_cases.png")
    plt.close()
    
    # ========== Visualize Cases with Least Improvement ==========
    fig, axes = plt.subplots(num_worst, 4, figsize=(16, 4*num_worst))
    
    if num_worst == 1:
        axes = axes.reshape(1, -1)
    
    for i, sample in enumerate(worst_improvement):
        # LR
        lr_np = F.interpolate(sample['lr'].unsqueeze(0), scale_factor=8, mode='nearest')[0].permute(1, 2, 0).numpy()
        lr_np = np.clip(lr_np, 0, 1)
        axes[i, 0].imshow(lr_np)
        axes[i, 0].set_title(f'LR Input (Sample #{sample["batch_idx"]})', fontsize=9)
        axes[i, 0].axis('off')
        
        # HR
        hr_np = sample['hr'].permute(1, 2, 0).numpy()
        hr_np = np.clip(hr_np, 0, 1)
        axes[i, 1].imshow(hr_np)
        axes[i, 1].set_title('HR Ground Truth', fontsize=9)
        axes[i, 1].axis('off')
        
        # Bicubic
        bicubic_np = sample['sr_bicubic'].permute(1, 2, 0).numpy()
        bicubic_np = np.clip(bicubic_np, 0, 1)
        axes[i, 2].imshow(bicubic_np)
        axes[i, 2].set_title('Bicubic SR', fontsize=9)
        axes[i, 2].axis('off')
        
        # Ours
        ours_np = sample['sr_ours'].permute(1, 2, 0).numpy()
        ours_np = np.clip(ours_np, 0, 1)
        axes[i, 3].imshow(ours_np)
        axes[i, 3].set_title(f'Ours (Δ: {sample["improvement_psnr"]:+.2f} dB)', fontsize=9, color='red')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'Top {num_worst} Cases with Least Improvement vs. Bicubic', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'worst_improvement_cases.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: worst_improvement_cases.png")
    plt.close()
    
    # ========== Statistics for Failure Cases ==========
    print("\n" + "="*70)
    print("FAILURE CASE ANALYSIS")
    print("="*70)
    
    print(f"\nWorst PSNR Cases:")
    for i, sample in enumerate(worst_psnr[:5]):
        print(f"  #{i+1}: Batch {sample['batch_idx']}, PSNR={sample['psnr']:.2f} dB, SSIM={sample['ssim']:.4f}")
    
    print(f"\nCases with Least Improvement over Bicubic:")
    for i, sample in enumerate(worst_improvement[:5]):
        print(f"  #{i+1}: Batch {sample['batch_idx']}, Δ={sample['improvement_psnr']:+.2f} dB (PSNR={sample['psnr']:.2f} dB)")
    
    # Check for regressions
    regressions = [s for s in all_samples if s['improvement_psnr'] < 0]
    print(f"\n⚠️  Regression Cases (worse than Bicubic): {len(regressions)} samples")
    
    if len(regressions) > 0:
        print(f"  Average regression: {np.mean([s['improvement_psnr'] for s in regressions]):.2f} dB")
    
    print(f"\n✓ Failure case analysis completed and saved to: {save_dir}")


print("✓ Failure case analysis function defined!")

In [ ]:
# ========== 9. EXECUTE COMPREHENSIVE EVALUATION WITH ALL VISUALIZATIONS ==========

print("\n" + "="*80)
print(" " * 15 + "🚀 COMPREHENSIVE MODEL EVALUATION WITH VISUALIZATIONS")
print("="*80)

# Create evaluation output directory
eval_output_dir = '/kaggle/working/RFB-ESRGAN-Output/evaluation'
os.makedirs(eval_output_dir, exist_ok=True)

print(f"\n📁 Evaluation results will be saved to: {eval_output_dir}")

# ========== PHASE 1: COMPARATIVE EVALUATION ==========
print("\n" + "="*80)
print("PHASE 1: COMPARATIVE EVALUATION (5 Models)")
print("="*80)

comparison_table, results, inference_times = comparative_evaluation(
    generator=generator,
    val_loader=val_loader,
    device=device,
    num_samples=100  # Evaluate on 100 samples
)

# ========== PHASE 2: COMPARATIVE VISUALIZATIONS ==========
print("\n" + "="*80)
print("PHASE 2: STATISTICAL COMPARISON VISUALIZATIONS")
print("="*80)

viz_dir = os.path.join(eval_output_dir, 'comparisons')
create_comparison_visualizations(results, inference_times, save_dir=viz_dir)

# Log visualizations to WandB
if wandb.run is not None:
    print("\n📤 Uploading comparison charts to WandB...")
    for img_name in ['metrics_comparison.png', 'metrics_distribution.png', 'radar_chart.png', 'quality_speed_tradeoff.png']:
        img_path = os.path.join(viz_dir, img_name)
        if os.path.exists(img_path):
            try:
                wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})
                print(f"  ✓ Uploaded: {img_name}")
            except Exception as e:
                print(f"  ⚠️  Could not upload {img_name}: {e}")
else:
    print("\n⚠️  WandB not initialized - skipping upload")

# ========== PHASE 3: 4-PANEL COMPARATIVE QUALITY GRIDS ==========
print("\n" + "="*80)
print("PHASE 3: 4-PANEL COMPARATIVE QUALITY GRIDS")
print("  Panel A: Low-Resolution Input (10m)")
print("  Panel B: Bicubic Interpolation")
print("  Panel C: RFB-ESRGAN Output (2.5m)")
print("  Panel D: Ground Truth")
print("="*80)

grid_dir = os.path.join(eval_output_dir, '4panel_grids')
generate_comprehensive_4panel_grid(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=grid_dir,
    num_samples=10
)

# ========== PHASE 4: AGRICULTURAL ROI FEATURE ANALYSIS ==========
print("\n" + "="*80)
print("PHASE 4: AGRICULTURAL ROI FEATURE ANALYSIS")
print("  • Crop Row Reconstruction")
print("  • Field Boundary Definition")
print("  • Texture Fidelity (Forest vs Water)")
print("="*80)

roi_agri_dir = os.path.join(eval_output_dir, 'agricultural_roi')
generate_agricultural_roi_analysis(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=roi_agri_dir,
    num_samples=5
)

# ========== PHASE 5: NDVI SPECTRAL CONSISTENCY ==========
print("\n" + "="*80)
print("PHASE 5: NDVI SPECTRAL CONSISTENCY ANALYSIS")
print("  Ensuring biological data (crop health) preservation")
print("="*80)

ndvi_dir = os.path.join(eval_output_dir, 'ndvi_spectral')
generate_ndvi_spectral_analysis(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=ndvi_dir,
    num_samples=5
)

# ========== PHASE 6: VISUAL QUALITY COMPARISON ==========
print("\n" + "="*80)
print("PHASE 6: DETAILED VISUAL QUALITY SAMPLES")
print("="*80)

quality_dir = os.path.join(eval_output_dir, 'quality_samples')
visualize_quality_comparison(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=quality_dir,
    num_samples=5
)

# Log quality samples to WandB
print("\n📤 Uploading quality samples to WandB...")
for i in range(1, 6):
    img_path = os.path.join(quality_dir, f'comparison_sample_{i}.png')
    if os.path.exists(img_path):
        try:
            if wandb.run is not None:
                wandb.log({f"eval/quality_sample_{i}": wandb.Image(img_path)})
        except Exception as e:
            print(f"  ⚠️  Could not upload quality sample {i}: {e}")

# ========== PHASE 7: ERROR & DIFFERENCE MAPS ==========
print("\n" + "="*80)
print("PHASE 7: PIXEL DIFFERENCE ERROR HEATMAPS")
print("="*80)

diff_dir = os.path.join(eval_output_dir, 'difference_maps')
visualize_difference_maps(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=diff_dir,
    num_samples=5
)

# Log difference maps to WandB
print("\n📤 Uploading error heatmaps to WandB...")
for i in range(1, 6):
    img_path = os.path.join(diff_dir, f'error_map_sample_{i}.png')
    if os.path.exists(img_path):
        try:
            if wandb.run is not None:
                wandb.log({f"eval/error_map_{i}": wandb.Image(img_path)})
        except Exception as e:
            print(f"  ⚠️  Could not upload error map {i}: {e}")

# ========== PHASE 8: CONVERGENCE ANALYSIS ==========
print("\n" + "="*80)
print("PHASE 8: MODEL CONVERGENCE ANALYSIS")
print("="*80)

convergence_dir = os.path.join(eval_output_dir, 'convergence')
analyze_model_convergence(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=convergence_dir
)

# Log convergence plots to WandB
print("\n📤 Uploading convergence analysis to WandB...")
for img_name in ['convergence_analysis.png', 'performance_trends.png']:
    img_path = os.path.join(convergence_dir, img_name)
    if os.path.exists(img_path):
        try:
            if wandb.run is not None:
                wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})
        except Exception as e:
            print(f"  ⚠️  Could not upload {img_name}: {e}")

# ========== PHASE 9: FAILURE CASE ANALYSIS ==========
print("\n" + "="*80)
print("PHASE 9: FAILURE CASE ANALYSIS")
print("="*80)

failure_dir = os.path.join(eval_output_dir, 'failure_cases')
analyze_failure_cases(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=failure_dir,
    num_worst=10
)

# Log failure cases to WandB
print("\n📤 Uploading failure case analysis to WandB...")
for img_name in ['worst_psnr_cases.png', 'worst_improvement_cases.png']:
    img_path = os.path.join(failure_dir, img_name)
    if os.path.exists(img_path):
        try:
            if wandb.run is not None:
                wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})
        except Exception as e:
            print(f"  ⚠️  Could not upload {img_name}: {e}")

# ========== FINAL SUMMARY ==========
print("\n" + "="*80)
print(" " * 20 + "✅ COMPREHENSIVE EVALUATION COMPLETE")
print("="*80)

print(f"\n📊 Summary of Complete Evaluation:")
print(f"\n  COMPARATIVE ANALYSIS:")
print(f"    • 5 models compared on 100 samples")
print(f"    • Statistical significance testing completed")
print(f"    • 4+ comparison charts generated")

print(f"\n  VISUAL OUTPUTS GENERATED:")
print(f"    • 10 × 4-Panel Comparative Grids (LR → Bicubic → Model → GT)")
print(f"    • 5 × Agricultural ROI Feature Analysis (Crop rows, boundaries, textures)")
print(f"    • 5 × NDVI Spectral Consistency Analysis (Biological data preservation)")
print(f"    • 5 × Detailed Quality Comparisons")
print(f"    • 5 × Pixel Difference Error Heatmaps")
print(f"    • Convergence distribution & trend analysis")
print(f"    • Top 10 failure cases identified")

print(f"\n  TOTAL VISUALIZATIONS: 30+ high-resolution images")

print(f"\n📁 All results saved to: {eval_output_dir}")
print(f"    ├── 4panel_grids/          (10 images)")
print(f"    ├── agricultural_roi/      (5 images)")
print(f"    ├── ndvi_spectral/         (5 images)")
print(f"    ├── quality_samples/       (5 images)")
print(f"    ├── difference_maps/       (5 images)")
print(f"    ├── comparisons/           (4 charts)")
print(f"    ├── convergence/           (2 plots)")
print(f"    └── failure_cases/         (2 plots)")

try:
    if wandb.run is not None:
        print(f"\n📤 All metrics and visualizations uploaded to WandB")
        print(f"   View at: {wandb.run.url}")
    else:
        print(f"\n⚠️  WandB not initialized - results saved locally only")
except Exception as e:
    print(f"\n⚠️  WandB not available: {e}")
    print(f"   Results saved locally to: {eval_output_dir}")


print("\n🎉 Comprehensive agricultural SR evaluation completed successfully!")print("="*80)